# Silver Layer - Aggregated Transactions

This notebook transforms PhonePe aggregated transaction JSON data from the Bronze layer into a clean, analytics-ready Silver Delta table.

## Source
- Bronze table: `workspace.bronze.phonepe_raw_json`
- Metadata table: `workspace.silver.phonepe_file_metadata`

## Target
- Silver table: `workspace.silver.aggregated_transactions`

## Processing
- Select aggregated transaction files
- Parse nested JSON using an explicit schema
- Extract transaction category, instrument type, count, and amount
- Preserve state, year, quarter, source path, checksum, and ingestion metadata
- Validate the transformed dataset

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    ArrayType,
    StringType,
    BooleanType,
    LongType,
    DoubleType
)

# Source tables
BRONZE_TABLE = "workspace.bronze.phonepe_raw_json"
METADATA_TABLE = "workspace.silver.phonepe_file_metadata"

# Target Silver table
SILVER_TABLE = "workspace.silver.aggregated_transactions"

# Expected aggregated transaction files:
# 34 country files + 1,224 state files
EXPECTED_SOURCE_FILES = 1_258

print(f"Bronze source : {BRONZE_TABLE}")
print(f"Metadata source: {METADATA_TABLE}")
print(f"Silver target : {SILVER_TABLE}")
print(f"Expected files: {EXPECTED_SOURCE_FILES:,}")

Bronze source : workspace.bronze.phonepe_raw_json
Metadata source: workspace.silver.phonepe_file_metadata
Silver target : workspace.silver.aggregated_transactions
Expected files: 1,258


In [0]:
payment_instrument_schema = StructType([
    StructField("type", StringType(), True),
    StructField("count", LongType(), True),
    StructField("amount", DoubleType(), True)
])

transaction_category_schema = StructType([
    StructField("name", StringType(), True),
    StructField(
        "paymentInstruments",
        ArrayType(payment_instrument_schema),
        True
    )
])

aggregated_transaction_schema = StructType([
    StructField("success", BooleanType(), True),
    StructField("code", StringType(), True),
    StructField(
        "data",
        StructType([
            StructField("from", LongType(), True),
            StructField("to", LongType(), True),
            StructField(
                "transactionData",
                ArrayType(transaction_category_schema),
                True
            )
        ]),
        True
    ),
    StructField("responseTimestamp", LongType(), True)
])

print("Aggregated transaction JSON schema created successfully.")

Aggregated transaction JSON schema created successfully.


In [0]:
bronze_df = spark.table(BRONZE_TABLE).alias("b")

transaction_metadata_df = (
    spark.table(METADATA_TABLE)
    .filter(
        (F.col("dataset_group") == "aggregated") &
        (F.col("domain") == "transaction")
    )
    .alias("m")
)

transaction_source_df = (
    transaction_metadata_df
    .join(
        bronze_df,
        F.col("m.source_path") == F.col("b.source_path"),
        "inner"
    )
    .select(
        F.col("m.relative_path"),
        F.col("m.file_scope"),
        F.col("m.state_slug"),
        F.col("m.year"),
        F.col("m.quarter"),
        F.col("m.period"),
        F.col("b.raw_json"),
        F.col("b.source_path"),
        F.col("b.source_commit"),
        F.col("b.content_sha256"),
        F.col("b.ingested_at_utc")
    )
)

actual_source_files = (
    transaction_source_df
    .select("source_path")
    .distinct()
    .count()
)

print(f"Expected source files: {EXPECTED_SOURCE_FILES:,}")
print(f"Actual source files  : {actual_source_files:,}")

assert actual_source_files == EXPECTED_SOURCE_FILES, (
    f"File-count validation failed: expected "
    f"{EXPECTED_SOURCE_FILES:,}, found {actual_source_files:,}"
)

print("Aggregated transaction source validation passed.")

Expected source files: 1,258
Actual source files  : 1,258
Aggregated transaction source validation passed.


In [0]:
parsed_transaction_df = (
    transaction_source_df
    .withColumn(
        "parsed_json",
        F.from_json(
            F.col("raw_json"),
            aggregated_transaction_schema
        )
    )
)

parse_summary = (
    parsed_transaction_df
    .select(
        F.count("*").alias("source_files"),
        F.sum(
            F.when(
                F.col("parsed_json.data.transactionData").isNull(),
                1
            ).otherwise(0)
        ).alias("files_without_transaction_data")
    )
    .first()
)

print(f"Source files                  : {parse_summary['source_files']:,}")
print(
    "Files without transaction data: "
    f"{parse_summary['files_without_transaction_data']:,}"
)

assert parse_summary["files_without_transaction_data"] == 0, (
    "JSON parsing validation failed: one or more files do not "
    "contain readable transactionData."
)

print("JSON parsing validation passed.")

Source files                  : 1,258
Files without transaction data: 0
JSON parsing validation passed.


In [0]:
silver_transactions_df = (
    parsed_transaction_df

    # One row for every transaction category
    .withColumn(
        "transaction_record",
        F.explode(F.col("parsed_json.data.transactionData"))
    )

    # One row for every payment instrument inside the category
    .withColumn(
        "payment_instrument",
        F.explode(F.col("transaction_record.paymentInstruments"))
    )

    # Create the final Silver columns
    .select(
        F.col("file_scope"),
        F.col("state_slug"),
        F.col("year").cast("int").alias("year"),
        F.col("quarter").cast("int").alias("quarter"),
        F.col("period"),

        F.trim(F.col("transaction_record.name"))
            .alias("transaction_category_raw"),

        F.trim(F.col("payment_instrument.type"))
            .alias("payment_instrument_type"),

        F.col("payment_instrument.count")
            .cast("long")
            .alias("transaction_count"),

        F.col("payment_instrument.amount")
            .cast("double")
            .alias("transaction_amount"),

        F.col("parsed_json.data.from")
            .alias("source_period_from_epoch_ms"),

        F.col("parsed_json.data.to")
            .alias("source_period_to_epoch_ms"),

        F.col("parsed_json.responseTimestamp")
            .alias("response_timestamp_epoch_ms"),

        F.col("relative_path"),
        F.col("source_path"),
        F.col("source_commit"),
        F.col("content_sha256"),
        F.col("ingested_at_utc")
    )

    # Standardized category for analysis across different years
    .withColumn(
        "transaction_category",
        F.when(
            F.lower(F.col("transaction_category_raw")).isin(
                "peer-to-peer payments",
                "p2p"
            ),
            F.lit("P2P")
        )
        .when(
            F.lower(F.col("transaction_category_raw")).isin(
                "recharge & bill payments",
                "utility",
                "utilities"
            ),
            F.lit("Utilities")
        )
        .when(
            F.lower(F.col("transaction_category_raw")).isin(
                "merchant payments",
                "financial services",
                "others",
                "retail",
                "business"
            ),
            F.lit("Business")
        )
        .otherwise(F.col("transaction_category_raw"))
    )

    # Put the standardized category beside the original category
    .select(
        "file_scope",
        "state_slug",
        "year",
        "quarter",
        "period",
        "transaction_category_raw",
        "transaction_category",
        "payment_instrument_type",
        "transaction_count",
        "transaction_amount",
        "source_period_from_epoch_ms",
        "source_period_to_epoch_ms",
        "response_timestamp_epoch_ms",
        "relative_path",
        "source_path",
        "source_commit",
        "content_sha256",
        "ingested_at_utc"
    )
)

print("Nested transaction records transformed successfully.")

Nested transaction records transformed successfully.


In [0]:
quality_result = (
    silver_transactions_df
    .agg(
        F.count("*").alias("total_rows"),

        F.sum(
            F.when(
                F.col("transaction_category_raw").isNull() |
                F.col("payment_instrument_type").isNull() |
                F.col("transaction_count").isNull(),
                1
            ).otherwise(0)
        ).alias("missing_required_values"),

        F.sum(
            F.when(
                F.col("transaction_count") < 0,
                1
            ).otherwise(0)
        ).alias("negative_transaction_counts"),

        F.sum(
            F.when(
                F.col("transaction_amount").isNull(),
                1
            ).otherwise(0)
        ).alias("missing_transaction_amounts")
    )
    .first()
)

duplicate_rows = (
    silver_transactions_df
    .groupBy(
        "source_path",
        "transaction_category_raw",
        "payment_instrument_type"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Total Silver rows           : {quality_result['total_rows']:,}")
print(
    f"Missing required values     : "
    f"{quality_result['missing_required_values']:,}"
)
print(
    f"Negative transaction counts : "
    f"{quality_result['negative_transaction_counts']:,}"
)
print(
    f"Missing transaction amounts : "
    f"{quality_result['missing_transaction_amounts']:,}"
)
print(f"Duplicate business keys     : {duplicate_rows:,}")

assert quality_result["total_rows"] > 0, "No transaction rows were created."

assert quality_result["missing_required_values"] == 0, (
    "Required-value validation failed."
)

assert quality_result["negative_transaction_counts"] == 0, (
    "Negative transaction-count validation failed."
)

assert duplicate_rows == 0, (
    "Duplicate business-key validation failed."
)

print("Silver transaction quality checks passed.")

Total Silver rows           : 3,773
Missing required values     : 0
Negative transaction counts : 0
Missing transaction amounts : 3,773
Duplicate business keys     : 0
Silver transaction quality checks passed.


In [0]:
amount_structure_check_df = (
    transaction_source_df
    .select(
        "year",
        F.get_json_object(
            F.col("raw_json"),
            "$.data.transactionData[0].paymentInstruments[0]"
        ).alias("sample_payment_instrument")
    )
    .groupBy("year")
    .agg(
        F.first(
            "sample_payment_instrument",
            ignorenulls=True
        ).alias("sample_payment_instrument")
    )
    .orderBy("year")
)

display(amount_structure_check_df)

year,sample_payment_instrument
2018,"{""type"":""TOTAL"",""count"":89921478}"
2019,"{""type"":""TOTAL"",""count"":703745511}"
2020,"{""type"":""TOTAL"",""count"":211773187}"
2021,"{""type"":""TOTAL"",""count"":2215589893}"
2022,"{""type"":""TOTAL"",""count"":367923036}"
2023,"{""type"":""TOTAL"",""count"":5929670663}"
2024,"{""type"":""TOTAL"",""count"":134770237}"
2025,"{""type"":""TOTAL"",""count"":10422073985}"
2026,"{""type"":""TOTAL"",""count"":1363409161}"


In [0]:
silver_transactions_final_df = (
    silver_transactions_df
    .drop("transaction_amount")
)

print(
    "Removed transaction_amount because it is absent "
    "from all files in this source version."
)

print(
    f"Final Silver columns: "
    f"{len(silver_transactions_final_df.columns)}"
)

display(
    silver_transactions_final_df
    .orderBy("year", "quarter", "file_scope", "state_slug")
    .limit(10)
)


Removed transaction_amount because it is absent from all files in this source version.
Final Silver columns: 17


file_scope,state_slug,year,quarter,period,transaction_category_raw,transaction_category,payment_instrument_type,transaction_count,source_period_from_epoch_ms,source_period_to_epoch_ms,response_timestamp_epoch_ms,relative_path,source_path,source_commit,content_sha256,ingested_at_utc
country,null,2018,1,2018-Q1,P2P,P2P,TOTAL,47101116,null,null,1785250510155,aggregated/transaction/country/india/2018/1.json,s3://phonepe-pulse-lakehouse-mohit-2026/raw/phonepe-pulse/source_commit=257bf38e3095020f9f516c4c72c10bdc348781b1/data/aggregated/transaction/country/india/2018/1.json,257bf38e3095020f9f516c4c72c10bdc348781b1,8518c9b68af16d8fba5b6d9e5d1bf367aaae9e40ad9db0cb03437f6ea5e6f041,2026-10-09T21:12:12.947Z
country,null,2018,1,2018-Q1,Retail,Business,TOTAL,89921478,null,null,1785250510155,aggregated/transaction/country/india/2018/1.json,s3://phonepe-pulse-lakehouse-mohit-2026/raw/phonepe-pulse/source_commit=257bf38e3095020f9f516c4c72c10bdc348781b1/data/aggregated/transaction/country/india/2018/1.json,257bf38e3095020f9f516c4c72c10bdc348781b1,8518c9b68af16d8fba5b6d9e5d1bf367aaae9e40ad9db0cb03437f6ea5e6f041,2026-10-09T21:12:12.947Z
country,null,2018,1,2018-Q1,Utility,Utilities,TOTAL,2955878,null,null,1785250510155,aggregated/transaction/country/india/2018/1.json,s3://phonepe-pulse-lakehouse-mohit-2026/raw/phonepe-pulse/source_commit=257bf38e3095020f9f516c4c72c10bdc348781b1/data/aggregated/transaction/country/india/2018/1.json,257bf38e3095020f9f516c4c72c10bdc348781b1,8518c9b68af16d8fba5b6d9e5d1bf367aaae9e40ad9db0cb03437f6ea5e6f041,2026-10-09T21:12:12.947Z
state,andaman-&-nicobar-islands,2018,1,2018-Q1,Utility,Utilities,TOTAL,136,null,null,1785250546335,aggregated/transaction/country/india/state/andaman-&-nicobar-islands/2018/1.json,s3://phonepe-pulse-lakehouse-mohit-2026/raw/phonepe-pulse/source_commit=257bf38e3095020f9f516c4c72c10bdc348781b1/data/aggregated/transaction/country/india/state/andaman-&-nicobar-islands/2018/1.json,257bf38e3095020f9f516c4c72c10bdc348781b1,9db7041d2800bf8884088c0b7488efddc3f99fb4e84c21e96497cee52e61eabc,2026-10-09T21:12:12.947Z
state,andaman-&-nicobar-islands,2018,1,2018-Q1,Retail,Business,TOTAL,6656,null,null,1785250546335,aggregated/transaction/country/india/state/andaman-&-nicobar-islands/2018/1.json,s3://phonepe-pulse-lakehouse-mohit-2026/raw/phonepe-pulse/source_commit=257bf38e3095020f9f516c4c72c10bdc348781b1/data/aggregated/transaction/country/india/state/andaman-&-nicobar-islands/2018/1.json,257bf38e3095020f9f516c4c72c10bdc348781b1,9db7041d2800bf8884088c0b7488efddc3f99fb4e84c21e96497cee52e61eabc,2026-10-09T21:12:12.947Z
state,andaman-&-nicobar-islands,2018,1,2018-Q1,P2P,P2P,TOTAL,2297,null,null,1785250546335,aggregated/transaction/country/india/state/andaman-&-nicobar-islands/2018/1.json,s3://phonepe-pulse-lakehouse-mohit-2026/raw/phonepe-pulse/source_commit=257bf38e3095020f9f516c4c72c10bdc348781b1/data/aggregated/transaction/country/india/state/andaman-&-nicobar-islands/2018/1.json,257bf38e3095020f9f516c4c72c10bdc348781b1,9db7041d2800bf8884088c0b7488efddc3f99fb4e84c21e96497cee52e61eabc,2026-10-09T21:12:12.947Z
state,andhra-pradesh,2018,1,2018-Q1,Retail,Business,TOTAL,6888562,null,null,1785250546335,aggregated/transaction/country/india/state/andhra-pradesh/2018/1.json,s3://phonepe-pulse-lakehouse-mohit-2026/raw/phonepe-pulse/source_commit=257bf38e3095020f9f516c4c72c10bdc348781b1/data/aggregated/transaction/country/india/state/andhra-pradesh/2018/1.json,257bf38e3095020f9f516c4c72c10bdc348781b1,aa8340515109bb470980454033633264d9fe4899d778b4d456108111db61af7e,2026-10-09T21:12:12.947Z
state,andhra-pradesh,2018,1,2018-Q1,Utility,Utilities,TOTAL,503773,null,null,1785250546335,aggregated/transaction/country/india/state/andhra-pradesh/2018/1.json,s3://phonepe-pulse-lakehouse-mohit-2026/raw/phonepe-pulse/source_commit=257bf38e3095020f9f516c4c72c10bdc348781b1/data/aggregated/transaction/country/india/state/andhra-pradesh/2018/1.json,257bf38e3095020f9f516c4c72c10bdc348781b1,aa8340515109bb470980454033633264d9fe4899d778b

In [0]:
(
    silver_transactions_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

print(f"Silver Delta table created: {SILVER_TABLE}")

Silver Delta table created: workspace.silver.aggregated_transactions


In [0]:
saved_transactions_df = spark.table(SILVER_TABLE)

saved_summary = (
    saved_transactions_df
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("source_path").alias("source_file_count"),
        F.min("period").alias("first_period"),
        F.max("period").alias("last_period"),
        F.sum("transaction_count").alias("total_transaction_count")
    )
    .first()
)

print(f"Saved rows             : {saved_summary['row_count']:,}")
print(f"Source files represented: {saved_summary['source_file_count']:,}")
print(f"First period           : {saved_summary['first_period']}")
print(f"Last period            : {saved_summary['last_period']}")
print(
    f"Total transaction count: "
    f"{saved_summary['total_transaction_count']:,}"
)

assert saved_summary["row_count"] == 3_773, (
    "Saved row-count validation failed."
)

assert saved_summary["source_file_count"] == EXPECTED_SOURCE_FILES, (
    "Not every source file is represented in the Silver table."
)

assert saved_summary["first_period"] == "2018-Q1"
assert saved_summary["last_period"] == "2026-Q2"

print("Final Silver Delta-table validation passed.")

Saved rows             : 3,773
Source files represented: 1,258
First period           : 2018-Q1
Last period            : 2026-Q2
Total transaction count: 874,879,725,414
Final Silver Delta-table validation passed.


In [0]:
category_evolution_df = (
    saved_transactions_df
    .groupBy(
        "year",
        "transaction_category_raw",
        "transaction_category"
    )
    .agg(
        F.sum("transaction_count").alias("transaction_count")
    )
    .orderBy(
        "year",
        F.desc("transaction_count")
    )
)

display(category_evolution_df)

year,transaction_category_raw,transaction_category,transaction_count
2018,Retail,Business,1092858184
2018,P2P,P2P,1048391554
2018,Utility,Utilities,76837728
2019,Retail,Business,3914840142
2019,P2P,P2P,3911319370
2019,Utility,Utilities,422687680
2020,Retail,Business,7286538538
2020,P2P,P2P,6984331854
2020,Utility,Utilities,1774462842
2021,Retail,Business,15890925342


## Silver Transaction Model — Completion Summary

The `workspace.silver.aggregated_transactions` Delta table was successfully created.

### Validation results

- Source files represented: **1,258**
- Silver records: **3,773**
- Data coverage: **2018-Q1 to 2026-Q2**
- Invalid required values: **0**
- Negative transaction counts: **0**
- Duplicate business keys: **0**
- Total recorded transaction count: **874,879,725,414**

### Source-data limitation

The pinned PhonePe Pulse source version does not contain transaction amount values. Therefore, this model retains transaction counts and excludes the completely empty amount column.

### Category standardization

Both the original and standardized category values are retained:

- `P2P` → `P2P`
- `Utility` → `Utilities`
- `Retail` → `Business`

This preserves source fidelity while enabling consistent longitudinal analytics and machine-learning features.